# Analisis Eksperimen Penalaan Hiperparameter (Hyperparameter Tuning)
### Modul Penalaan Hiperparameter Mandiri - Ummu NLP Lab

Notebook ini digunakan untuk melaksanakan proses **Penalaan Hiperparameter (*Hyperparameter Tuning*)** dengan metode **Grid Search** di atas data validasi. Model-model yang ditala meliputi:
1. **Multinomial Naïve Bayes** (Parameter: smoothing $\alpha$)
2. **Support Vector Machine (SVM)** (Parameter: regularisasi $C$, kernel, dan $\gamma$)
3. **IndoBERT** (Parameter: *learning rate* dan *batch size*)

Metrik yang digunakan sebagai acuan pemilihan parameter terbaik adalah **Macro F1-Score** pada data validasi.


## 1. Persiapan Lingkungan & Pemuatan Dependensi
Melakukan impor pustaka analisis data, Scikit-Learn, PyTorch, dan Transformers, serta mengonfigurasi deteksi CUDA GPU.


In [ ]:
import os
import re
import time
import json
import random
import numpy as np
import pandas as pd

# Scikit-Learn Machine Learning Libraries
from sklearn.model_selection import ParameterGrid
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# Deep Learning Libraries (PyTorch & Transformers)
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import logging as transformers_logging

# Meredam warning yang tidak penting
transformers_logging.set_verbosity_error()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[GPU Device Status] Active Device: {DEVICE} (CUDA Available: {torch.cuda.is_available()})")

def set_seed(seed=42):
    """Enforces exact reproducibility across runs."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 2. Pemuatan Dataset SmSA IndoNLU Resmi
Memuat dataset dari repositori resmi IndoNLP untuk kebutuhan pelatihan dan validasi tuning.


In [ ]:
TRAIN_URL = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv"
VALID_URL = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/valid_preprocess.tsv"

print("Memuat dataset dari repositori IndoNLP...")
df_train_raw = pd.read_csv(TRAIN_URL, sep='\t', header=None, names=['text', 'label'])
df_valid_raw = pd.read_csv(VALID_URL, sep='\t', header=None, names=['text', 'label'])

print(f"-> Ukuran Dataset Train : {len(df_train_raw)} baris")
print(f"-> Ukuran Dataset Valid : {len(df_valid_raw)} baris")

## 3. Kamus Stopwords, Slang Words, dan Pipeline Preprocessing
Mengonfigurasi kamus stopwords, slang words, serta alur normalisasi teks ulasan secara bertahap.


In [ ]:
INDONESIAN_STOPWORDS = set([
    'yang', 'di', 'dan', 'itu', 'dengan', 'untuk', 'dari', 'ke', 'ini', 'adalah',
    'bisa', 'ada', 'pada', 'juga', 'saya', 'kami', 'mereka', 'dia', 'anda', 'kamu',
    'akan', 'telah', 'sudah', 'sedang', 'dalam', 'oleh', 'olehnya', 'atau', 'tetapi',
    'namun', 'hanya', 'saja', 'jika', 'kalau', 'karena', 'sehingga', 'maka', 'tentang',
    'seperti', 'terhadap', 'secara', 'kembali', 'kemudian', 'lalu', 'setelah',
    'sebelum', 'ketika', 'saat', 'sementara', 'bagi', 'sangat', 'amat',
    'paling', 'lebih', 'kurang', 'terlalu', 'banyak', 'beberapa', 'semua',
    'tiap', 'setiap', 'bukan', 'tidak', 'tak', 'belum', 'jangan', 'bagaimana', 'apa',
    'siapa', 'dimana', 'kapan', 'mengapa', 'kenapa', 'ya', 'oh', 'sih', 'lah', 'deh',
    'kah', 'pun', 'kok', 'punya', 'buat', 'ialah'
])

SLANG_WORDS_DICT = {
    'yg': 'yang', 'dgn': 'dengan', 'utk': 'untuk', 'sy': 'saya', 'tdk': 'tidak',
    'gak': 'tidak', 'ga': 'tidak', 'tp': 'tetapi', 'bgt': 'sangat', 'bkn': 'bukan',
    'klo': 'kalau', 'pake': 'pakai', 'pas': 'saat', 'sdg': 'sedang', 'hub': 'hubung',
    'org': 'orang', 'krn': 'karena', 'lu': 'kamu', 'gw': 'saya', 'aja': 'saja',
    'sm': 'sama', 'bener': 'benar', 'udh': 'sudah', 'udah': 'sudah', 'jd': 'jadi',
    'gpp': 'tidak apa-apa', 'bs': 'bisa', 'bbrp': 'beberapa', 'msh': 'masih', 'dr': 'dari'
}

def preprocess_text(text):
    """Pembersihan teks ulasan lengkap (Klasik/Tradisional)."""
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    tokens = re.findall(r'\b[a-zA-Z0-9]+\b', text)
    cleaned = []
    for tok in tokens:
        norm_tok = SLANG_WORDS_DICT.get(tok, tok)
        if norm_tok not in INDONESIAN_STOPWORDS:
            cleaned.append(norm_tok)
    return " ".join(cleaned)

def preprocess_text_minimal(text):
    """Pembersihan teks ulasan minimalis (IndoBERT)."""
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    tokens = re.findall(r'\b[a-zA-Z0-9]+\b', text)
    cleaned = [SLANG_WORDS_DICT.get(tok, tok) for tok in tokens]
    return " ".join(cleaned)

print("Menjalankan preprocessing data...")
X_train_classic = [preprocess_text(t) for t in df_train_raw['text']]
X_valid_classic = [preprocess_text(t) for t in df_valid_raw['text']]
y_train = df_train_raw['label'].tolist()
y_valid = df_valid_raw['label'].tolist()

X_train_bert = [preprocess_text_minimal(t) for t in df_train_raw['text']]
X_valid_bert = [preprocess_text_minimal(t) for t in df_valid_raw['text']]
print("[OK] Preprocessing selesai!")

## 4. Custom PyTorch Dataset untuk IndoBERT
Menginisiasi pembentuk dataset (dataset builder) untuk menghasilkan representasi tensor input_ids dan attention_mask.


In [ ]:
class IndonesianTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length, label_map):
        self.texts = texts
        self.labels = [label_map[l] for l in labels]
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

## 5. Fungsi Penghitung Metrik Evaluasi
Menghitung tingkat keberhasilan prediksi klasifikasi menggunakan metrik Macro F1-Score.


In [ ]:
def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "macro_f1": f1
    }

## 6. Fungsi Penalaan Hiperparameter Model Klasik (`tune_classical_models`)
Melakukan pencarian kombinasi parameter untuk Multinomial Naïve Bayes dan SVM di atas data validasi klasik secara sistematis.


In [ ]:
def tune_classical_models(X_train, X_val, y_train, y_val):
    print("=== HYPERPARAMETER TUNING: CLASSICAL MODELS ===\n")
    
    # Ekstraksi fitur TF-IDF (min_df=5)
    min_df_val = 5 if len(X_train) >= 25 else 1
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=min_df_val)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec = vectorizer.transform(X_val)
    
    # 1. Tuning Naive Bayes
    nb_grid = {'alpha': [0.1, 0.5, 1.0, 1.5, 2.0]}
    best_nb_f1 = -1
    best_nb_params = None
    nb_results = []
    
    for g in ParameterGrid(nb_grid):
        model = MultinomialNB(alpha=g['alpha'])
        model.fit(X_train_vec, y_train)
        y_pred = model.predict(X_val_vec)
        metrics = calculate_metrics(y_val, y_pred)
        f1 = metrics['macro_f1']
        nb_results.append({"alpha": g['alpha'], "macro_f1": f1})
        if f1 > best_nb_f1:
            best_nb_f1 = f1
            best_nb_params = g
            
    print("[Hasil Tuning Naive Bayes]")
    print(pd.DataFrame(nb_results).to_string(index=False))
    print(f'-> Parameter Terbaik NB: {best_nb_params} | Macro F1: {best_nb_f1:.2%}\n')
    
    # 2. Tuning SVM
    svm_grid = {
        'C': [0.1, 1.0, 10.0],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    }
    best_svm_f1 = -1
    best_svm_params = None
    svm_results = []
    
    for g in ParameterGrid(svm_grid):
        model = SVC(C=g['C'], kernel=g['kernel'], gamma=g['gamma'])
        model.fit(X_train_vec, y_train)
        y_pred = model.predict(X_val_vec)
        metrics = calculate_metrics(y_val, y_pred)
        f1 = metrics['macro_f1']
        svm_results.append({
            "C": g['C'], "kernel": g['kernel'], "gamma": g['gamma'], "macro_f1": f1
        })
        if f1 > best_svm_f1:
            best_svm_f1 = f1
            best_svm_params = g
            
    print("[Hasil Tuning SVM]")
    print(pd.DataFrame(svm_results).to_string(index=False))
    print(f'-> Parameter Terbaik SVM: {best_svm_params} | Macro F1: {best_svm_f1:.2%}\n')
    
    return {
        "naive_bayes": best_nb_params,
        "svm": best_svm_params
    }

## 7. Fungsi Penalaan Hiperparameter IndoBERT (`tune_bert_model`)
Melakukan fine-tuning model IndoBERT pada lingkungan GPU untuk mengevaluasi kombinasi learning_rate dan batch_size (tanpa menggunakan model tiruan/simulasi).


In [ ]:
def tune_bert_model(X_train, X_val, y_train, y_val):
    print("=== HYPERPARAMETER TUNING: INDOBERT ===\n")
    
    if DEVICE == "cpu":
        raise RuntimeError("Tuning IndoBERT memerlukan akselerasi GPU (CUDA). Harap ganti runtime Google Colab Anda ke T4 GPU.")
        
    bert_grid = {
        'learning_rate': [2e-5, 5e-5],
        'batch_size': [8, 16]
    }
    classes = sorted(list(set(y_train + y_val)))
    label_map = {name: i for i, name in enumerate(classes)}
    inv_label_map = {i: name for name, i in label_map.items()}
    num_labels = len(classes)
    model_name = "indobenchmark/indobert-base-p1"
    
    best_f1 = -1
    best_params = None
    results = []
    
    for g in ParameterGrid(bert_grid):
        set_seed(42)
        
        print(f"Menguji Parameter: LR={g['learning_rate']} | Batch Size={g['batch_size']}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels, ignore_mismatched_sizes=True)
        model.to(DEVICE)
        
        train_dataset = IndonesianTextDataset(X_train, y_train, tokenizer, 128, label_map)
        eval_dataset = IndonesianTextDataset(X_val, y_val, tokenizer, 128, label_map)
        train_loader = DataLoader(train_dataset, batch_size=g['batch_size'], shuffle=True)
        eval_loader = DataLoader(eval_dataset, batch_size=g['batch_size'])
        
        optimizer = AdamW(model.parameters(), lr=g['learning_rate'])
        
        # Latih 2 epoch untuk mempercepat pencarian parameter
        for epoch in range(2):
            model.train()
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                targets = batch['label'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                
        model.eval()
        y_pred_idx = []
        with torch.no_grad():
            for batch in eval_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1)
                y_pred_idx.extend(preds.cpu().tolist())
                
        y_pred = [inv_label_map[idx] for idx in y_pred_idx]
        metrics = calculate_metrics(y_val, y_pred)
        f1 = metrics['macro_f1']
        results.append({
            "learning_rate": g['learning_rate'],
            "batch_size": g['batch_size'],
            "macro_f1": f1
        })
        if f1 > best_f1:
            best_f1 = f1
            best_params = g
            
    print(f'\n[Hasil Tuning IndoBERT]')
    print(pd.DataFrame(results).to_string(index=False))
    print(f'-> Parameter Terbaik IndoBERT: {best_params} | Macro F1: {best_f1:.2%}\n')
    return best_params

## 8. Eksekusi Grid Search Penalaan Hiperparameter
Mengeksekusi fungsi penalaan parameter klasik dan deep learning di atas data validasi secara berurutan.


In [ ]:
best_classical = tune_classical_models(X_train_classic, X_valid_classic, y_train, y_valid)
best_bert = tune_bert_model(X_train_bert, X_valid_bert, y_train, y_valid)

print("=== PROSES PENALAAN SELESAI ===")
print(f"-> NB   : {best_classical['naive_bayes']}")
print(f"-> SVM  : {best_classical['svm']}")
print(f"-> BERT : {best_bert}")